# fast

In [2]:
# bl average

import csv

def bl():
    bl = []
    with open("BL.csv", "r") as f:
        reader = csv.reader(f)
        next(reader)
        for i in reader:
            bl.append(float(i[1]))
    return sum(bl)/len(bl)
print(bl())

30.714285714285715


In [2]:
""" Unit conversion
    -> velocity_second
"""
import csv
import os


folder = "trajectories_tidy_group_removed/02"
foldero = "velocity_second/02"
files = os.listdir(folder)

for file in files:
    nfile = os.path.basename(file)+"_v_second.csv"
    writer = csv.writer(open(os.path.join(foldero, nfile), "w", newline=''))
    writer.writerow(["time", "velocity", "velocity(cm/s)"])
    lines = {}
    reader = csv.DictReader(open(os.path.join(folder, file), "r"))
    for line in reader:
        if int(line["frame"]) > 1500*30 or float(line["acceleration"]) > 10000:
            continue
        k = int(int(line["frame"]) / 30)
        if k not in lines:
            lines[k] = [
                [float(line["velocity"])],
                [float(line["velocity"]) * 1.875]
            ]
        else:
            lines[k][0].append(float(line["velocity"]))
            lines[k][1].append(float(line["velocity"]) * 1.875)
    for k in lines:
        writer.writerow(
            [k, sum(lines[k][0])/len(lines[k][0]), sum(lines[k][1])/len(lines[k][1])]
        )

### 

folder = "trajectories_tidy_group_removed/07"
foldero = "velocity_second/07"
files = os.listdir(folder)

for file in files:
    nfile = os.path.basename(file)+"_v_second.csv"
    fo = open(os.path.join(foldero, nfile), "w", newline='')
    writer = csv.writer(fo)
    writer.writerow(["time", "velocity", "velocity(cm/s)"])
    lines = {}
    reader = csv.DictReader(open(os.path.join(folder, file), "r"))
    for line in reader:
        if int(line["frame"]) > 1500*30 or float(line["acceleration"]) > 10000:
            continue
        k = int(int(line["frame"]) / 30)
        if k not in lines:
            lines[k] = [
                [float(line["velocity"])],
                [float(line["velocity"]) * 1.875]
            ]
        else:
            lines[k][0].append(float(line["velocity"]))
            lines[k][1].append(float(line["velocity"]) * 1.875)
    nlines = []
    for k in lines:
        nlines.append(
            [k, sum(lines[k][0])/len(lines[k][0]), sum(lines[k][1])/len(lines[k][1])]
        )
    writer.writerows(nlines)
    fo.close()

In [ ]:
# calculate
# 1px/f = 1.875cm/s

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from scipy.signal import savgol_filter
import os
import glob
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.gridspec as gridspec
import csv
import matplotlib as mpl


def graph(file_path, ucrit) :
    mpl.rcParams['font.size'] = 20
    dd = pd.DataFrame()

    csv_file = file_path
    df = pd.read_csv(csv_file).query('frame <= 45000').query('acceleration < 10000')    # 25min = 25*60*30frame
    dd['velocity'] = df['velocity'] * 1.875
    
    dd['ucrit'] = [ucrit for _ in range(len(dd))]

    fig = plt.figure(figsize=(18, 5), dpi=300)

    # Calculate smoothed velocity
    smoothed_velocity = None
    try:
        # Apply Savitzky-Golay filter for smoothing
        if len(df) > 10:  # Need enough points for smoothing
            plt.plot(df['frame'], dd['velocity'], '#ff7e0d', linewidth=0.8, alpha=0.8, label="velocity")
        else:
            plt.text(0.5, 0.5, 'Insufficient data for smoothing', 
                    horizontalalignment='center', verticalalignment='center',
                    transform=plt.transAxes, fontsize=14)
    except Exception as e:
        print(f"Warning: Could not create smoothed velocity curve: {e}")
        plt.text(0.5, 0.5, f'Error creating smoothed curve: {str(e)}', 
                horizontalalignment='center', verticalalignment='center',
                transform=plt.transAxes, fontsize=12)

    plt.plot(df['frame'], dd['ucrit'], "#000000", linewidth=0.8, alpha=0.8, label="ucrit")

    # Set labels and title
    plt.xlabel('Time (seconds)')
    plt.ylabel('Velocity (cm/s)')
    plt.title('Smoothed Fish Velocity Over Time')

    # Add grid and legend
    plt.grid(True, alpha=0.3)
    plt.legend()

def caculate(file_path, ucrit):
    num = 0
    fast = 0
    with open(file_path) as f:
        reader = csv.DictReader(f)
        for line in reader:
            if int(line["time"]) < 15*60:   # 15min = 15*60sec
                continue
            if float(line["velocity(cm/s)"]) > ucrit:
                fast+=1
            num+=1
    print(f"{num}\t{fast}\t{fast/num}")
    return fast/num

folder = "velocity_second/02"
files = os.listdir(folder)
a = []
for file in files:
    a.append(caculate(os.path.join(folder, file), 3*2))
    # graph(os.path.join(folder, file), 9)
print(sum(a)/len(a))
###
folder = "velocity_second/07"
files = os.listdir(folder)
a = []
for file in files:
    a.append(caculate(os.path.join(folder, file), 3*2))
    # graph(os.path.join(folder, file), 9)
print(sum(a)/len(a))

601	55	0.09151414309484193
601	74	0.12312811980033278
601	78	0.129783693843594
601	70	0.11647254575707154
601	198	0.32945091514143093
601	23	0.03826955074875208
601	241	0.40099833610648916
601	51	0.08485856905158069
601	199	0.33111480865224624
601	51	0.08485856905158069
601	91	0.15141430948419302
601	36	0.059900166389351084
601	81	0.13477537437603992
0.15973377703826955
601	268	0.4459234608985025
601	151	0.2512479201331115
601	119	0.19800332778702162
601	95	0.15806988352745424
601	57	0.09484193011647254
601	73	0.12146422628951747
601	191	0.3178036605657238
601	102	0.16971713810316139
601	172	0.28618968386023297
601	80	0.13311148086522462
601	14	0.02329450915141431
601	201	0.33444259567387685
601	147	0.24459234608985025
601	46	0.07653910149750416
601	157	0.2612312811980033
601	161	0.26788685524126454
601	195	0.324459234608985
601	85	0.14143094841930118
601	64	0.1064891846921797
601	201	0.33444259567387685
601	99	0.16472545757071547
0.212186039141114
